# Logistic Regression Exercise Solutions

---

Solutions to the exercises of `notebooks/Python/02_EAIF_Logistic_Regression.ipynb`. The notebook is self-contained: the first code cells rebuild the data, the stratified split, the undersampled training set and the two models exactly as in the main notebook, so every exercise can be run on its own. For each exercise you find the task, one code cell, and the result with its interpretation.

Numbers quoted in the result cells come from a local run of this notebook; Colab may differ in the last digit.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score
%matplotlib inline
np.random.seed(42)


In [ ]:
# Rebuild the data, the split, the undersampled training set and the models exactly as in the main notebook
banking_url = "https://raw.githubusercontent.com/umatter/EDFB/main/data/banking.csv"
dataset = pd.read_csv(banking_url).drop(columns=['duration', 'pdays', 'age', 'campaign', 'previous'])
num_var = dataset.drop(columns=['y']).select_dtypes([np.number]).columns
dataset_dummy = pd.get_dummies(dataset, drop_first=True, dtype=float)
col_to_drop = ['emp_var_rate', 'cons_price_idx', 'euribor3m', 'nr_employed', 'loan_unknown']
dataset = dataset_dummy.drop(columns=col_to_drop)
num_var = [v for v in num_var if v in dataset.columns]
X = dataset.drop(columns=['y'])
y = dataset['y'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_train, X_test = X_train.copy(), X_test.copy()
scaler = preprocessing.StandardScaler().fit(X_train[num_var])   # scaler fit on the training rows only
X_train[num_var] = scaler.transform(X_train[num_var])
X_test[num_var] = scaler.transform(X_test[num_var])

# Undersample the training set only
train_data = X_train.copy()
train_data['y'] = y_train
train_1 = train_data[train_data['y'] == 1]
train_0 = train_data[train_data['y'] == 0]
train_0_small = train_0.sample(n=2 * len(train_1), random_state=0)
train_bal = pd.concat([train_1, train_0_small]).sample(frac=1, random_state=0)
X_train_bal = train_bal.drop(columns=['y'])
y_train_bal = train_bal['y'].values
sampling_ratio = len(train_0_small) / len(train_0)

# Models
lpm_model = LinearRegression().fit(X_train_bal, y_train_bal)
y_test_pred_lpm = lpm_model.predict(X_test)
logit_model = LogisticRegression(C=np.inf, solver='lbfgs', max_iter=1000, tol=1e-8).fit(X_train_bal, y_train_bal)
y_test_predicted_prob_logit = logit_model.predict_proba(X_test)[:, 1]
p_test = 1 / (1 + np.exp(-(logit_model.decision_function(X_test) + np.log(sampling_ratio))))   # deployment probabilities

odds_table = pd.DataFrame({'feature': X_train_bal.columns,
                           'log_odds_coef': logit_model.coef_[0],
                           'odds_ratio': np.exp(logit_model.coef_[0])})
odds_table = odds_table.reindex(odds_table['log_odds_coef'].abs().sort_values(ascending=False).index).reset_index(drop=True)

print(f"Test set: {len(y_test)} clients, share subscribed {y_test.mean():.3f}")
print(f"Sampling ratio {sampling_ratio:.3f}; AUC on the test set {roc_auc_score(y_test, p_test):.3f}")


## Exercise 1: LPM versus logit predictions

**Task:** Scatter the LPM probabilities against the logit probabilities on the test set with the 45-degree line; report the share of clients on which the two models agree at threshold 0.5, the largest absolute difference, and where along the probability range the models disagree.


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test_pred_lpm, y_test_predicted_prob_logit, alpha=0.3, s=10)
plt.plot([0, 1], [0, 1], 'r--', label='45-degree line')
plt.xlabel('LPM predicted probability')
plt.ylabel('Logit predicted probability')
plt.title('LPM versus logit on the test set')
plt.legend()
plt.show()

lpm_class = (y_test_pred_lpm >= 0.5).astype(int)
logit_class = (y_test_predicted_prob_logit >= 0.5).astype(int)
agreement = (lpm_class == logit_class).mean()
diff = np.abs(y_test_pred_lpm - y_test_predicted_prob_logit)
i_max = diff.argmax()
print(f"Same 0/1 classification at 0.5 for {100 * agreement:.1f}% of the test clients")
print(f"Largest disagreement: LPM {y_test_pred_lpm[i_max]:.3f} vs logit {y_test_predicted_prob_logit[i_max]:.3f} (difference {diff[i_max]:.3f})")

# Where do they disagree? Mean absolute difference by range of the logit probability
bins = pd.cut(y_test_predicted_prob_logit, [0, 0.2, 0.4, 0.6, 0.8, 1.0])
print(pd.DataFrame({'bin': bins, 'abs_diff': diff}).groupby('bin', observed=True)['abs_diff'].agg(['mean', 'count']).round(3))


**Result.** In this run the two models make the same 0/1 classification for 99.7% of the test clients. The largest disagreement is a client with LPM value 1.032 and logit probability 0.946 (difference 0.087): the LPM runs above 1 where the logit flattens out. The mean absolute difference is below 0.01 between 0 and 0.4, where most clients sit, and largest (0.02 to 0.05) above 0.6, where the S-curve bends and the straight line cannot follow. For classification at 0.5 the choice of model barely matters; for probabilities near the boundaries it does.

## Exercise 2: Threshold sweep

**Task:** For thresholds 0.01 to 0.50 on the deployment probabilities `p_test`, compute the number of calls, precision and recall; plot them; show the rows for 0.05, 0.10 and 0.20 and describe the trade-off.


In [ ]:
thresholds = np.round(np.arange(0.01, 0.51, 0.01), 2)
rows = []
for t in thresholds:
    call = (p_test >= t).astype(int)
    rows.append({'threshold': t, 'calls': call.sum(),
                 'precision': precision_score(y_test, call, zero_division=0),
                 'recall': recall_score(y_test, call)})
sweep = pd.DataFrame(rows)

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(sweep['threshold'], sweep['precision'], label='precision')
ax1.plot(sweep['threshold'], sweep['recall'], label='recall')
ax1.set_xlabel('threshold on p_test')
ax1.set_ylabel('precision / recall')
ax1.legend(loc='center right')
ax2 = ax1.twinx()
ax2.plot(sweep['threshold'], sweep['calls'], color='grey', linestyle=':', label='calls')
ax2.set_ylabel('number of calls (of %d test clients)' % len(y_test))
plt.title('Threshold sweep on the test set')
plt.show()

display(sweep[sweep['threshold'].isin([0.05, 0.10, 0.20])].round(3))


**Result.** Lowering the threshold trades precision for recall through the number of calls. In this run: at 0.05 the bank calls 6,512 of the 8,238 test clients (79%), reaches 91% of the subscribers, and 13% of the calls end in a sale; at 0.10 it calls 3,995 (48%), reaches 70%, and precision is 16%; at 0.20 it calls 374 (5%), reaches 24%, and 59% of the calls succeed. The model ranks clients (the AUC is about 0.70), but few clients have a high probability, so precision only rises once the call list becomes very short.

## Exercise 3: Expected profit and the optimal threshold

**Task:** With a €5 call cost and €100 revenue per subscription, compute the profit per 10,000 clients for every threshold, compare with calling everyone, derive the rule `p > cost / revenue`, and repeat with a €20 call cost.


In [ ]:
contact_cost = 5
subscription_revenue = 100
total_customers = 10000
n_test = len(y_test)
thresholds = np.round(np.arange(0.01, 0.51, 0.01), 2)

def profit_per_10k(threshold, cost, revenue=subscription_revenue):
    call = p_test >= threshold
    tp = (call & (y_test == 1)).sum()
    return (revenue * tp - cost * call.sum()) / n_test * total_customers

plt.figure(figsize=(8, 5))
for cost in [contact_cost, 20]:
    profits = pd.Series([profit_per_10k(t, cost) for t in thresholds], index=thresholds)
    everyone = (subscription_revenue * (y_test == 1).sum() - cost * n_test) / n_test * total_customers
    best_t = profits.idxmax()
    rule_t = cost / subscription_revenue
    print(f"Call cost €{cost}: break-even p = cost / revenue = {rule_t:.2f}")
    print(f"  call everyone (no model):     €{everyone:>9,.0f} per 10,000 clients")
    print(f"  best threshold {best_t:.2f}:          €{profits.max():>9,.0f} per 10,000 clients, calling {(p_test >= best_t).mean():.0%} of clients")
    print(f"  rule threshold {rule_t:.2f}:          €{profit_per_10k(rule_t, cost):>9,.0f} per 10,000 clients, calling {(p_test >= rule_t).mean():.0%} of clients")
    plt.plot(thresholds, profits.values, label=f'call cost €{cost}')
plt.axhline(0, color='grey', linewidth=0.5)
plt.xlabel('threshold on p_test')
plt.ylabel('profit per 10,000 clients (€)')
plt.title('Expected profit by threshold')
plt.legend()
plt.show()


**Result.** Call cost EUR 5: the break-even probability is 5 / 100 = 0.05. Calling everyone earns about EUR 62,600 per 10,000 clients; the best threshold (0.07 in this run) earns about EUR 63,300, and the rule threshold 0.05 about EUR 62,700. The model adds almost nothing, and that is the honest result: with a base rate of 11.3% and a break-even of 5%, nearly every client is worth a call, so the decision hardly depends on the ranking. Call cost EUR 20: the break-even is 0.20. Calling everyone now loses about EUR 87,000 per 10,000 clients, while the rule threshold 0.20 earns about EUR 17,500 (calling 5% of clients) and the best threshold (0.17 in this run) about EUR 18,000. The rule in one sentence: call a client when the expected revenue p x EUR 100 exceeds the cost of the call, that is when p > cost / revenue. The empirical optimum lies close to that value; the flat profit curve around it is sampling noise on a finite test set. The rule needs the deployment probabilities `p_test`: the probabilities of the undersampled fit are inflated (their mean is 0.30 instead of 0.11), so applying p > 0.05 to them would call everyone and p > 0.20 would still call far too many. The value of the model depends on the economics of the campaign, not on the AUC alone.

## Exercise 4: Odds ratios in business terms

**Task:** Extract the odds ratios of `poutcome_success`, `contact_telephone` and `cons_conf_idx`, explain each in one sentence, and sort the model's features into what the manager controls, what she can target on, and what she cannot influence.


In [ ]:
three = odds_table[odds_table['feature'].isin(['poutcome_success', 'contact_telephone', 'cons_conf_idx'])]
display(three.round(3))

groups = {
    'controls directly': ['contact_telephone'],
    'can target on, cannot change': [f for f in X_train.columns if f.split('_')[0] in ('marital', 'education', 'housing', 'loan', 'poutcome')],
    'cannot influence (macro)': ['cons_conf_idx'],
}
for g, feats in groups.items():
    print(f"{g}: {', '.join(feats)}")


**Result.** `poutcome_success`, odds ratio about 10.9 in this run: a client who subscribed in the previous campaign has about 11 times the odds of subscribing again, other things equal (the unit is having the characteristic). `contact_telephone`, odds ratio about 0.38: a client reached on a landline has 38% of the odds of a client reached on a mobile phone, so about 60% lower odds. `cons_conf_idx`, odds ratio about 1.15 per standard deviation of consumer confidence (about 4.6 index points): one standard deviation more confidence raises the odds of a sale by about 15%. Groups: (a) the manager controls the contact channel (`contact_telephone`), with the caveat that the coefficient may partly reflect who is reachable by landline rather than the channel itself; (b) marital status, education, housing and personal loans and the outcome of the previous campaign are client characteristics she can use to decide whom to call but cannot change; (c) consumer confidence is a macro variable she cannot influence, except by timing the campaign.

## Exercise 5: Three clients

**Task:** Encode clients A, B and C with the columns of `X_train`, compute their deployment probabilities, rank them, and apply the €5/€100 rule (call if p > 0.05).


In [ ]:
customers = pd.DataFrame(0.0, index=['A', 'B', 'C'], columns=X_train.columns)
# A: married, university degree, housing loan, no personal loan, cellular, previous campaign success, confidence average
customers.loc['A', ['marital_married', 'education_university.degree', 'housing_yes', 'poutcome_success']] = 1
# B: single, high school, no loans, landline, never contacted before, confidence one SD below average
customers.loc['B', ['marital_single', 'education_high.school', 'contact_telephone', 'poutcome_nonexistent']] = 1
customers.loc['B', 'cons_conf_idx'] = -1
# C: divorced (reference), basic.9y, housing loan and personal loan, cellular (reference), previous failure (reference), confidence average
customers.loc['C', ['education_basic.9y', 'housing_yes', 'loan_yes']] = 1

logodds = logit_model.decision_function(customers) + np.log(sampling_ratio)
result = pd.DataFrame({'p_undersampled_fit': logit_model.predict_proba(customers)[:, 1],
                       'p_deploy': 1 / (1 + np.exp(-logodds))}, index=customers.index)
result['call_at_5_over_100'] = result['p_deploy'] > 0.05
display(result.sort_values('p_deploy', ascending=False).round(3))


**Result.** Deployment probabilities in this run: A 0.665, C 0.111, B 0.051. Client A, with a previous success, is far above the 5% break-even and is called first. Client C sits near the base rate (11%) and is worth a call at EUR 5 per call. Client B is right at the break-even (0.051 here; the R twin, with its different random split, puts the same client at 0.049, just below it): the expected profit of the call is about zero, so at EUR 5 the decision is a coin toss and at EUR 20 it is clearly no. The column `p_undersampled_fit` shows why the correction matters: on the uncorrected scale B looks like a 17% prospect and C like a 33% prospect.